In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

In [ ]:
from marine_qc import (
    combine_qc_results,
    do_iquam_track_check,
    do_multiple_sequential_check,
    do_spike_check,
)

In [ ]:
from data import get_sequential_data

# How to use quality control checks with sequential reports

We need some text!!!

In [ ]:
data = get_sequential_data()
data

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 8))

axs[0, 0].plot(data["lon"], data["lat"], marker="o", linestyle="-")
axs[0, 0].set_title("Ship – latitude/longitude track")
axs[0, 0].set_xlabel("Longitude (°)")
axs[0, 0].set_ylabel("Latitude (°)")
axs[0, 0].grid(True)

axs[0, 1].plot(data["vsi"], marker="o", linestyle="-")
axs[0, 1].set_title("Ship – speed")
axs[0, 1].set_xlabel("Time")
axs[0, 1].set_ylabel("Speed (km/h)")
axs[0, 1].grid(True)

axs[1, 0].plot(data["dsi"], marker="o", linestyle="-")
axs[1, 0].set_title("Ship – direction")
axs[1, 0].set_xlabel("Time")
axs[1, 0].set_ylabel("Direction (°)")
axs[1, 0].grid(True)

axs[1, 1].plot(data["sst"], marker="o", linestyle="-")
axs[1, 1].set_title("Ship – sea surface temperature")
axs[1, 1].set_xlabel("Time")
axs[1, 1].set_ylabel("Temperature (°C)")
axs[1, 1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
qc_spike = do_spike_check(
    value=data.sst,
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    max_gradient_space=0.5,
    max_gradient_time=1.0,
    delta_t=1.0,
    n_neighbours=5,
)
pd.DataFrame({"lat": data.lat, "lon": data.lon, "date": data.date, "sst": data.sst, "qc_spike": qc_spike})

In [ ]:
qc_track = do_iquam_track_check(
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    speed_limit=15.0,
    delta_d=1.11,
    delta_t=0.01,
    n_neighbours=5,
)
pd.DataFrame({"lat": data.lat, "lon": data.lon, "date": data.date, "qc_track": qc_track})

In [ ]:
qc_dict = {
    "spike_check": {
        "func": "do_spike_check",
        "names": {
            "value": "sst",
            "lat": "lat",
            "lon": "lon",
            "date": "date",
        },
        "arguments": {
            "max_gradient_space": 0.5,
            "max_gradient_time": 1.0,
            "delta_t": 1.0,
            "n_neighbours": 5,
        },
    },
    "iquam_track_check": {
        "func": "do_iquam_track_check",
        "names": {
            "lat": "lat",
            "lon": "lon",
            "date": "date",
        },
        "arguments": {
            "speed_limit": 15.0,
            "delta_d": 1.11,
            "delta_t": 0.01,
            "n_neighbours": 5,
        },
    },
}

In [ ]:
qc_multi = do_multiple_sequential_check(
    data,
    qc_dict=qc_dict,
    groupby="ship_id",
    return_method="failed",
)
qc_multi

In [ ]:
qc_flag = combine_qc_results(qc_multi)

In [ ]:
pd.DataFrame({**data, "qc_flag": qc_flag})